# 🌲 Stacking & Voting Ensembles — Solutions Notebook

**Difficulty**: ⭐⭐ Intermediate  
**Time**: ~40 mins  
**Complete, verified reference implementation.**

---


## 🎯 Section 1: Overview

Voting combines predictions of distinct models via majority or probability averaging. Stacking uses a meta-learner trained on out-of-fold base predictions.

### Stacking Meta-Learner:
$$y_{meta} = g(f_1(x), f_2(x), \dots, f_M(x))$$


## 🔧 Section 2: Implementation from Scratch


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Setup complete! ✅')

In [ ]:
from sklearn.model_selection import KFold
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

class StackingClassifierFromScratch:
    def __init__(self, base_models, meta_model, n_folds=5):
        self.base_models = base_models
        self.meta_model = meta_model
        self.n_folds = n_folds
        
    def fit(self, X, y):
        self.fitted_base_models = [[] for _ in self.base_models]
        oof_preds = np.zeros((X.shape[0], len(self.base_models)))
        
        kf = KFold(n_splits=self.n_folds, shuffle=True, random_state=42)
        
        for train_idx, val_idx in kf.split(X, y):
            X_tr, y_tr = X[train_idx], y[train_idx]
            X_val = X[val_idx]
            
            for i, model in enumerate(self.base_models):
                cloned_model = clone(model)
                cloned_model.fit(X_tr, y_tr)
                oof_preds[val_idx, i] = cloned_model.predict(X_val)
                self.fitted_base_models[i].append(cloned_model)
                
        # Train meta model
        self.meta_model.fit(oof_preds, y)
        return self

    def predict(self, X):
        meta_features = np.zeros((X.shape[0], len(self.base_models)))
        for i, model_list in enumerate(self.fitted_base_models):
            fold_preds = np.array([m.predict(X) for m in model_list])
            meta_features[:, i] = np.mean(fold_preds, axis=0)
        return self.meta_model.predict(meta_features)


In [ ]:
X = np.random.rand(100, 4)
y = np.random.randint(0, 2, 100)
sc = StackingClassifierFromScratch([RandomForestClassifier(n_estimators=5), DecisionTreeClassifier(max_depth=3)], LogisticRegression())
sc.fit(X, y)
print('Stacking predictions:', sc.predict(X)[:5])


## 📦 Section 3: Library Implementation


In [ ]:
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

stacking = StackingClassifier(
    estimators=[('rf', RandomForestClassifier()), ('gbm', GradientBoostingClassifier())],
    final_estimator=LogisticRegression(),
    cv=5
)
stacking.fit(X_train, y_train)


## ❓ Section 4: Interview Questions


### Q1: Why must out-of-fold (OOF) predictions be used when training a Stacking Meta-Learner?
**Answer**: If base model predictions on the training data are fed directly to the meta-learner, the meta-learner will suffer severe target leakage and overfit to whichever base model overfits the training set most.


### Q2: What is the difference between Hard Voting and Soft Voting?
**Answer**: Hard voting uses simple majority rule across predicted class labels. Soft voting averages the predicted class probabilities from each base estimator, giving higher weight to confident predictions.
